# Exercise — Configure Governed Access in Lake Formation

**Trailhead Provisions** needs four personas to have exactly the access below on the `customer`
table. Configure the Lake Formation grants with the provided helpers, then run the verification
cell to prove enforcement by querying as each persona in Athena. See `INSTRUCTIONS.md`.

| Persona | Intended access | LF mode |
|---|---|---|
| `co_marketing_analyst` | Customer **minus** `full_name`, `email`, `phone` | column-level |
| `co_cs_rep` | All columns, **rows where `region='EU'`** only | row-level |
| `co_data_steward` | Everything tagged `governed=true` | tag-based (LF-TBAC) |
| `co_auditor` | Schema + tags only; **no row data** | metadata-only |

> This exercise runs exclusively against **live AWS Glue, Lake Formation, and Amazon Athena** (no local fallback).

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog, S3 tables, and Lake Formation enforcement are seeded.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement if needed
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, verify_step7, PERSONAS

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__}")
print(f"  Glue DB:          {gc.database}")
print(f"  Region:           {gc.region}")
print(f"  Athena workgroup: {gc.athena_workgroup}")
print(f"  Results bucket:   {gc.results_bucket}")

## 1. Reset & confirm the starting state (provided)
Clear any grants from a previous run, then confirm every persona starts with no access.

In [ ]:
gc.reset()
gc.grants_summary()

## 2. Apply the four Lake Formation grants
Use the provided helpers:
- **Column-level (`co_marketing_analyst`)**: `gc.grant_columns("co_marketing_analyst", "customer", exclude=["full_name", "email", "phone"])`
- **Row-level (`co_cs_rep`)**: `gc.create_row_filter(name, "customer", "region = 'EU'")` + `gc.grant_rows("co_cs_rep", "customer", name)`
- **Tag-based (`co_data_steward`)**: First tag `customer` as `governed=true` with `gc.assign_tag('customer', 'governed', 'true')` (the governed table in this exercise), then grant `co_data_steward` access by tag with `gc.grant_by_tag('co_data_steward', 'governed', 'true')`.
- **Metadata-only (`co_auditor`)**: `gc.grant_describe_only("co_auditor")`

In [ ]:
# TODO(you): Apply all four grants below.
# 1) column-level: analyst sees customer minus all three PII columns (exclude=["full_name", "email", "phone"])

# 2) row-level: cs_rep sees only EU rows (region = 'EU')

# 3) tag-based: First tag `customer` as `governed=true` with `gc.assign_tag('customer', 'governed', 'true')`
#    (the governed table in this exercise), then grant `co_data_steward` access by tag with
#    `gc.grant_by_tag('co_data_steward', 'governed', 'true')`.

# 4) metadata-only: auditor gets DESCRIBE, no SELECT

gc.grants_summary()

## 3. Verify enforcement (provided)
Queries the governed table as each persona and asserts the intended outcome. Every check must read **PASS**.

In [ ]:
results = verify_step7(gc)
print("All checks pass?", (results["result"] == "PASS").all())
results

## 4. Policy spec
Write a short policy spec documenting the intended access for each persona and the LF grant type used.

TODO(you): Write your policy spec here. Address each persona: what access they get, why, and which Lake Formation mode enforces it.

---
## 5. Infrastructure as Code — Read and Explain

Open `terraform/governed-data-product/main.tf` (provided, read-only). This Terraform module
packages everything you just did by hand into a repeatable, version-controlled declaration.

**Your task**: Read the module and answer the questions below. You do NOT need to run
`terraform apply` — this is a reading-comprehension exercise.

### Questions

1. **What is the governance chain?** List the 4 AWS resources the module creates, in order,
   and explain how each one connects to the next to enforce governed access.

2. **Why is there NO `s3:GetObject` on the data bucket?** What does the consumer role use
   instead to read data, and what enforces the column/row restrictions?

3. **How does `excluded_column_names` in the `aws_lakeformation_permissions` resource relate
   to what you did in Section 2?** Which of your four grants does this Terraform resource
   implement?

4. **If a new domain team wants to add a `product` data product with PII column
   `supplier_contact_email`, what values would they pass to this module?** (List the
   variable values.)

TODO(you): Answer each of the 4 questions above. Reference specific resources and lines from the Terraform module in your answers.